In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cvxpy as cp
from functions import AlgPenDisc
from functions import AlgCCS
from functions import AlgRemez
from functions import AlgCS
from functions import AlgProxPen
from functions import linear


In [ ]:
def test(m,d,seed=0):
    np.random.seed(seed)
    res=np.zeros([4,5,d])
    k=0
    for d in np.arange(0,d):
        Y=[(0,1)]
        n=m
        A=np.zeros((2*n+1,n+1))
        b=np.ones((2*n+1))
        for i in range(n):
            A[i][i]=-1
            A[n+i][i]=1
            A[-1][-1]=-1
            b[-1]=0
        
        X=[(linear,[A,b])]
        X1=[(-1,1) for i in range(n)]
        Y0=[[np.random.rand()]]
        x0=np.zeros(n+1)
        alpha=np.random.rand(n+1)
        beta=np.random.rand(n+1)
        gamma=np.random.rand(n+1)
        delta=np.random.rand(n+1)
        for i in range(n):
            delta[i]=delta[i]/np.sum(delta[:n])
        def fcp(x):
            return x[-1]

        def gcp(x,y):
            return -x[n]+cp.sum([((i+alpha[i])*x[i-1]-(i+beta[i])/n-np.sin(2*np.pi*y[0]+gamma[i]+i))**2  for i in np.arange(1,n+1)])

        def df(x):
            z=np.zeros(np.size(x))
            z[-1]=1
            return(z)

        def dg(x,y):
            z=np.zeros(np.size(x))
            z[-1]=-1
            for i in np.arange(1,n+1):
                z[i-1]=(i+alpha[i])*2*((i+alpha[i])*x[i-1]-(i+beta[i])/n-np.sin(2*np.pi*y+gamma[i]+i))
            #print(z)
            return(z)


        def g(x,y):
            return -x[n]+cp.sum([((i+alpha[i])*x[i-1]-(i+beta[i])/n-np.sin(2*np.pi*y[0]+gamma[i]+i))**2 for i in np.arange(1,n+1)])
        def g1(x,y):
            return -x[n]+cp.sum([((i+alpha[i])*x[i-1]-(i+beta[i])/n-np.sin(2*np.pi*y+gamma[i]+i))**2 for i in np.arange(1,n+1)])
        
        rl=AlgProxPen(x0,Y0,fcp,gcp,g,X,Y,f=fcp,df=df,rho=1,drop_cuts=True,gamma=1)
        res[0][1][k]=rl[2]
        res[1][1][k]=rl[3]
        res[2][1][k]=rl[4]
        res[3][1][k]=rl[5]

        
        rl=AlgCCS(x0,Y0,fcp,gcp,g,X,Y,Ub=40*n)
        res[0][2][k]=rl[2]
        res[1][2][k]=rl[3]
        res[2][2][k]=rl[4]
        res[3][2][k]=rl[4]
        
        rl=AlgCS(x0,Y0,fcp,gcp,g,X,Y)
        res[0][3][k]=rl[2]
        res[1][3][k]=rl[3]
        res[2][3][k]=rl[4]
        res[3][3][k]=rl[4]
        
        rl=AlgRemez(x0,Y0,fcp,gcp,g,X,Y)
        res[0][4][k]=rl[2]
        res[1][4][k]=rl[3]
        res[2][4][k]=rl[4]
        res[3][4][k]=rl[4]
        
        k+=1
    
    
   
    print("exp_sin_dim"+str(m)+"_seed"+str(seed))
    np.save("exp_sin_dim"+str(m)+"_seed"+str(seed), res)
    return res

def tables(res):
    print("times mean/median ProxLinDrop",np.mean(res[0][1]),np.median(res[0][1]))
    print("times mean/median CCS",np.mean(res[0][2]),np.median(res[0][2]))
    print("times mean/median CS",np.mean(res[0][3]),np.median(res[0][3]))
    print("times mean/median Remez",np.mean(res[0][4]),np.median(res[0][4]))

    print("feasibility mean/median ProxLinDrop",np.mean(res[1][1]),np.median(res[1][1]))
    print("feasibility mean/median CCS",np.mean(res[1][2]),np.median(res[1][2]))
    print("feasibility mean/median CS",np.mean(res[1][3]),np.median(res[1][3]))
    print("feasibility mean/median Remez",np.mean(res[1][4]),np.median(res[1][4]))

    print("iterations k mean/median ProxLinDrop",np.mean(res[3][1]),np.median(res[3][1]))
    print("iterations k mean/median CCS",np.mean(res[3][2]),np.median(res[3][2]))
    print("iterations k mean/median CS",np.mean(res[3][3]),np.median(res[3][3]))
    print("iterations k mean/median Remez",np.mean(res[3][4]),np.median(res[3][4]))

    print("calls vf mean/median ProxLinDrop",np.mean(res[2][1]),np.median(res[2][1]))
    print("calls vf mean/median CCS",np.mean(res[2][2]),np.median(res[2][2]))
    print("calls vf mean/median CS",np.mean(res[2][3]),np.median(res[2][3]))
    print("calls vf mean/median Remez",np.mean(res[2][4]),np.median(res[2][4]))

    plt.legend(["ProxLinDrop", "CCS","CS","Remez"])
    labels=["ProxLinDrop", "CCS","CS","Remez"]
    d=[res[0][1],res[0][2],res[0][3],res[0][4]]
    plt.boxplot(d,showfliers=False,showmeans=True, meanline=True, tick_labels=labels)
    

In [ ]:
#seed=np.random.randint(100000)
seed=50099
print(seed)
m=50
iters=100
res=test(m,iters,seed)

In [ ]:
res1=np.load("exp_sin_dim"+str(m)+"_seed"+str(seed)+".npy")
tables(res1)